# BDDK Erişim ve Vintaj Fizibilitesi — Ders Kitabı Notebooku

Bu notebook, BDDK haftalık taşıt kredisi serisinin ekonomik değerini değil, **as-of kullanılabilirliğini** sınar. Bir serinin bugün geçmişe uzanması, geçmiş tahmin originlerinde aynı değerlerin bilindiği anlamına gelmez. Revize olabilen serilerde ilk-yayım vintajı yoksa rolling-origin değerlendirme görünmez gelecek bilgisi kullanabilir.

## Okuma hedefleri

Bu notebook sonunda şu ayrımları kurabilmelisiniz:

1. Güncel tarihsel seri ile ilk-yayım vintajı arasındaki fark.
2. Erişim kapısının ekonomik mekanizma ve performans kapısından neden önce geldiği.
3. Eksik vintajın neden güncel değerle doldurulamayacağı.
4. Ön-kayıtlı `GEÇTİ / KOŞULLU / KALDI` hükmünün mekanik uygulanışı.

## Ön-kayıtlı örneklem

Sonuç görülmeden beş tarih kilitlendi: serinin ilk haftası, 2019 ve 2022'nin ilk haftaları, Model 10/11 analiz sınırındaki son hafta ve 2026-08-08 itibarıyla en güncel hafta. Ek tarih seçmek ve eksik vintajı başka haftayla ikame etmek yasaktı.

In [1]:
import pandas as pd

karsilastirma = pd.DataFrame([
    {'referans_hafta': '2014-01-03', 'guncel_milyon_tl': 8613.848, 'vintaj_milyon_tl': None, 'yayin_gecikmesi_gun': None},
    {'referans_hafta': '2019-01-04', 'guncel_milyon_tl': 6506.118, 'vintaj_milyon_tl': None, 'yayin_gecikmesi_gun': None},
    {'referans_hafta': '2022-01-07', 'guncel_milyon_tl': 12983.300, 'vintaj_milyon_tl': None, 'yayin_gecikmesi_gun': None},
    {'referans_hafta': '2025-04-25', 'guncel_milyon_tl': 64040.300, 'vintaj_milyon_tl': None, 'yayin_gecikmesi_gun': None},
    {'referans_hafta': '2026-07-31', 'guncel_milyon_tl': 42112.122, 'vintaj_milyon_tl': 42112.122, 'yayin_gecikmesi_gun': 6},
])
karsilastirma['delta_mutlak'] = (karsilastirma['guncel_milyon_tl'] - karsilastirma['vintaj_milyon_tl']).abs()
karsilastirma['delta_yuzde'] = karsilastirma['delta_mutlak'] / karsilastirma['vintaj_milyon_tl'].abs() * 100
karsilastirma

,referans_hafta,guncel_milyon_tl,vintaj_milyon_tl,yayin_gecikmesi_gun,delta_mutlak,delta_yuzde
0,2014-01-03,8613.848,NaN,NaN,NaN,NaN
1,2019-01-04,6506.118,NaN,NaN,NaN,NaN
2,2022-01-07,12983.300,NaN,NaN,NaN,NaN
3,2025-04-25,64040.300,NaN,NaN,NaN,NaN
4,2026-07-31,42112.122,42112.122,6.0,0.0,0.0


## Karar kapısının mekanik uygulanışı

`KALDI` için tek bir koşul yeterlidir: ilk tarih vintajının bulunmaması, vintaj erişiminin 3/5'in altında kalması, herhangi bir deltada %15 aşımı, kimlik/ücret gereksinimi veya yayın gecikmesinin sınırlandırılamaması. Burada hem ilk tarih yoktur hem vintaj erişimi yalnız 1/5'tir.

In [2]:
vintaj_sayisi = int(karsilastirma['vintaj_milyon_tl'].notna().sum())
ilk_tarih_vintaj_var = bool(pd.notna(karsilastirma.loc[0, 'vintaj_milyon_tl']))
olculen_delta = karsilastirma['delta_yuzde'].dropna()
delta_ge_15 = bool((olculen_delta >= 15).any())
delta_5_15 = int(((olculen_delta >= 5) & (olculen_delta < 15)).sum())
gecikme_tam_dogrulandi = bool(karsilastirma['yayin_gecikmesi_gun'].notna().all())

if (not ilk_tarih_vintaj_var) or (vintaj_sayisi < 3) or delta_ge_15 or (not gecikme_tam_dogrulandi):
    hukum = 'KALDI'
elif (vintaj_sayisi < 5) or (1 <= delta_5_15 <= 2):
    hukum = 'KOŞULLU'
elif delta_5_15 > 2:
    hukum = 'KALDI'
else:
    hukum = 'GEÇTİ'

{
    'vintaj_sayisi': vintaj_sayisi,
    'ilk_tarih_vintaj_var': ilk_tarih_vintaj_var,
    'delta_ge_15': delta_ge_15,
    'gecikme_tam_dogrulandi': gecikme_tam_dogrulandi,
    'hukum': hukum,
}

{'vintaj_sayisi': 1,
 'ilk_tarih_vintaj_var': False,
 'delta_ge_15': False,
 'gecikme_tam_dogrulandi': False,
 'hukum': 'KALDI'}

## Neyi kanıtladık, neyi kanıtlamadık?

**Kanıtlanan:** Canlı BDDK serisi ücretsiz/kimliksiz erişilebilir, 657 hafta kapsar ve beş seçili haftanın güncel değerini taşır. Resmî bağlantı zincirinde eski ilk-yayım dosya arşivi bulunamadı.

**Kanıtlanmayan:** Taşıt kredisi bakiyesinin yön tahmininde faydasız olduğu. Bu aşamada hedefle ilişki, oracle tavanı veya model performansı ölçülmedi. `KALDI`, sinyale değil erişim sözleşmesine aittir.

## Yanlış yorumlama kontrol listesi

- 657 haftalık tarih kapsamı, 657 ilk-yayım vintajı demek değildir.
- Son haftadaki %0 delta, eski dönemlerde revizyon olmadığına genellenemez.
- Eksik dört delta sıfır değildir; **hesaplanamadı**.
- Canlı uç noktanın tarih filtresi, sürümlenmiş arşiv anlamına gelmez.
- `KALDI` sonucu hedefi, sınıf sözleşmesini veya ekonomik mekanizmayı değiştirmez.
- Üçüncü taraf arşiv denenmemesi bir unutma değil, ön-kayıtlı kapsam sınırıdır.

## Kaynaklar

- [BDDK Haftalık Bülten — Gelişmiş Gösterim](https://www.bddk.org.tr/BultenHaftalik/tr/Gelismis)
- [BDDK Haftalık Bülten ana sayfası](https://www.bddk.org.tr/Veri/Detay/158)
- [Haftalık Bülten metaverisi](https://www.bddk.org.tr/BultenDosyalari/Home/Index/Haftalik-MetaVeri)
- [Veri yayımlama takvimi](https://www.bddk.org.tr/Veri/Detay/71)
- [Revizyon politikası](https://www.bddk.org.tr/BultenDosyalari/Home/Index/Haftalik-RevizyonPolitikas%C4%B1)
- [BDDK veri yayınları SSS](https://www.bddk.org.tr/Sss/Liste/110)

## Sonuç

**KALDI:** BDDK taşıt kredisi serisi mevcut resmî web erişimiyle as-of uyumlu geriye dönük feature üretimine alınamaz. Bir sonraki adım otomatik başlamaz; resmî/kurumsal vintaj temini veya farklı bir öncü bilgi ailesi için Pusula ve kullanıcı kararı gerekir.